### Audio Transcription

To transcribe audio, we'll use the `SpeechRecognition` library, which supports various speech recognition engines and APIs. We'll also use `pydub` to handle audio file conversions if needed.

In [ ]:
# Install necessary libraries
!pip install SpeechRecognition pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 57.2 MB/s eta 0:00:00


After installing the libraries, you can use the following code to transcribe an audio file. Make sure your audio file is in a supported format (like WAV). If it's in another format (e.g., MP3), `pydub` can help convert it.

**To use this code:**
1. Upload your audio file (e.g., `my_audio.wav` or `my_audio.mp3`) to your Colab environment by clicking the folder icon on the left panel and then the upload icon.
2. Replace `'path/to/your/audio_file.wav'` with the actual path to your uploaded audio file.

In [ ]:
import speech_recognition as sr
from pydub import AudioSegment
from pydub.silence import split_on_silence

# Path to your audio file
audio_file_path = 'youtube_audio.wav.wav' # Corrected path after yt-dlp download

# If your audio is not WAV, you might need to convert it first
# For example, if it's an MP3:
# try:
#     audio = AudioSegment.from_mp3(audio_file_path)
#     audio.export('temp_audio.wav', format='wav')
#     audio_file_path = 'temp_audio.wav'
# except Exception as e:
#     print(f"Could not convert audio to WAV: {e}")
#     # Handle cases where the file might already be WAV or another supported format

recognizer = sr.Recognizer()

try:
    with sr.AudioFile(audio_file_path) as source:
        print("Reading audio file...")
        audio_data = recognizer.record(source)  # Read the entire audio file

        print("Transcribing audio...")
        # Use Google Web Speech API for transcription (requires internet connection)
        text = recognizer.recognize_google(audio_data)

        print("\n--- Transcription ---")
        print(text)
        print("---------------------")

except sr.UnknownValueError:
    print("Speech Recognition could not understand audio")
except sr.RequestError as e:
    print(f"Could not request results from Google Speech Recognition service; {e}")
except FileNotFoundError:
    print(f"Error: Audio file not found at {audio_file_path}. Please check the path.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Error: Audio file not found at youtube_audio.wav.wav. Please check the path.


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


### Transcribing with OpenAI's Whisper

Since the previous attempt with Google Speech Recognition had a connection issue, we can try using OpenAI's Whisper model, which can run locally and is known for its high accuracy.

In [ ]:
# Install OpenAI's Whisper library
!pip install -q openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import whisper
import os
import torch

# Load the Whisper model
# You can choose different model sizes: 'tiny', 'base', 'small', 'medium', 'large'
# 'large' is generally the most accurate.
print("Loading Whisper model...")

# Explicitly set device to 'cpu' to avoid CUDA out of memory errors, as GPU memory is insufficient.
device = "cpu"
print(f"Using device: {device}")

# Load the 'tiny' model to minimize memory usage, specifying the device
model = whisper.load_model("tiny", device=device)
print("Whisper model loaded.")

# Path to the downloaded audio file
audio_file_path = 'youtube_audio.wav.wav'

if os.path.exists(audio_file_path):
    print(f"Transcribing audio with Whisper from: {audio_file_path}")
    # Transcribe the audio
    result = model.transcribe(audio_file_path)

    print("\n--- Whisper Transcription ---")
    print(result["text"])
    print("-----------------------------")
else:
    print(f"Error: Audio file not found at {audio_file_path}. Please ensure it was downloaded correctly.")

Loading Whisper model...
Using device: cpu


100%|█████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 76.5MiB/s]


Whisper model loaded.
Error: Audio file not found at youtube_audio.wav.wav. Please ensure it was downloaded correctly.


### Save Transcription to CSV

Now, let's save the transcribed text into a CSV file. We'll use the `pandas` library for this.

In [ ]:
import pandas as pd

# Extract segments from the result, which include start, end, and text
segments_data = []
for segment in result['segments']:
    segments_data.append({
        'start': segment['start'],
        'end': segment['end'],
        'text': segment['text'].strip() # Strip whitespace for cleaner text
    })

# Create a DataFrame from the segments data
df_transcription = pd.DataFrame(segments_data)

# Define the output CSV filename
csv_output_filename = 'youtube_transcription.csv'

# Save the DataFrame to a CSV file
df_transcription.to_csv(csv_output_filename, index=False)

print(f"Transcription saved to '{csv_output_filename}'")

# Display the entire DataFrame to confirm all lines
display(df_transcription)

NameError: name 'result' is not defined

The CSV file `youtube_transcription.csv` is now available in your Colab environment. You can download it by clicking the folder icon on the left sidebar and then locating the file.

### Download Audio from YouTube

To transcribe audio from a YouTube video, we first need to download the audio track. We'll use `yt-dlp` for this, which is a powerful command-line program to download videos and audio from YouTube and other sites. `ffmpeg` is often required for audio conversions during this process.

In [ ]:
# Install yt-dlp and ffmpeg (if not already installed)
!pip install yt-dlp
!apt-get install ffmpeg -y

In [ ]:
import yt_dlp
import os

youtube_url = 'https://www.youtube.com/watch?v=SxgaGXcQ8ZY' # The provided YouTube link
output_filename = 'youtube_audio.wav'

ydl_opts = {
    'format': 'bestaudio/best',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'wav',
        'preferredquality': '192',
    }],
    'outtmpl': output_filename,
    'noplaylist': True, # Ensure only single video is downloaded if playlist is linked
}

print(f"Downloading audio from: {youtube_url}")
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([youtube_url])

print(f"Audio downloaded to: {output_filename}")

# Update the audio_file_path for transcription
audio_file_path = output_filename


Now that the audio is downloaded, I will modify the previous transcription cell to use this newly downloaded audio file for transcription.